In [1]:
# imports
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import geopandas as gpd

In [2]:
nuclear_plants_data = Path.cwd().parent.joinpath('raw_data', 'nrc_reactor_locations', 'eia860_2024_nuclear_plant_locations.csv')

In [3]:
df = pd.read_csv(nuclear_plants_data)
df.head()

,Utility ID,Utility Name,Plant Code,Plant Name,Street Address,City,State,Zip,County,Latitude,...,Grid Voltage 2 (kV),Grid Voltage 3 (kV),Energy Storage,Natural Gas LDC Name,Natural Gas Pipeline Name 1,Natural Gas Pipeline Name 2,Natural Gas Pipeline Name 3,Pipeline Notes,Natural Gas Storage,Liquefied Natural Gas Storage
0,18642,Tennessee Valley Authority,46,Browns Ferry,Shaw Rd. PO Box 2000,Decatur,AL,35609,Limestone,34.70420,...,,,N,NaN,NaN,NaN,NaN,NaN,N,NaN
1,55951,Constellation Nuclear,204,Clinton Power Station,Rt. 54 West,Clinton,IL,61727,DeWitt,40.17190,...,,,N,NaN,NaN,NaN,NaN,NaN,N,X
2,20893,Wolf Creek Nuclear Optg Corp,210,Wolf Creek Generating Station,1550 Oxen Lane N.E.,Burlington,KS,66839,Coffey,38.23926,...,,,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,20160,Energy Northwest,371,Columbia Generating Station,76 N Power Plant Loop,Richland,WA,99352,Benton,46.47110,...,,,N,NaN,NaN,NaN,NaN,NaN,X,X
4,5221,Dominion Energy Nuclear Conn Inc,566,Millstone,314 Rope Ferry Road,Waterford,CT,6385,New London,41.31070,...,,,N,NaN,NaN,NaN,NaN,NaN,X,X


In [4]:
print(df.shape)
print(df.columns)

(56, 42)
Index(['Utility ID', 'Utility Name', 'Plant Code', 'Plant Name',
       'Street Address', 'City', 'State', 'Zip', 'County', 'Latitude',
       'Longitude', 'NERC Region', 'Balancing Authority Code',
       'Balancing Authority Name', 'Name of Water Source',
       'Primary Purpose (NAICS Code)', 'Regulatory Status', 'Sector',
       'Sector Name', 'FERC Cogeneration Status',
       'FERC Cogeneration Docket Number', 'FERC Small Power Producer Status',
       'FERC Small Power Producer Docket Number',
       'FERC Exempt Wholesale Generator Status',
       'FERC Exempt Wholesale Generator Docket Number', 'Ash Impoundment?',
       'Ash Impoundment Lined?', 'Ash Impoundment Status',
       'Transmission or Distribution System Owner',
       'Transmission or Distribution System Owner ID',
       'Transmission or Distribution System Owner State', 'Grid Voltage (kV)',
       'Grid Voltage 2 (kV)', 'Grid Voltage 3 (kV)', 'Energy Storage',
       'Natural Gas LDC Name', 'Natural Gas 

In [5]:
df.dtypes

Utility ID                                           int64
Utility Name                                        object
Plant Code                                           int64
Plant Name                                          object
Street Address                                      object
City                                                object
State                                               object
Zip                                                  int64
County                                              object
Latitude                                           float64
Longitude                                          float64
NERC Region                                         object
Balancing Authority Code                            object
Balancing Authority Name                            object
Name of Water Source                                object
Primary Purpose (NAICS Code)                         int64
Regulatory Status                                   obje

# Cleaning

In [6]:
df.isna().sum()

Utility ID                                          0
Utility Name                                        0
Plant Code                                          0
Plant Name                                          0
Street Address                                      0
City                                                0
State                                               0
Zip                                                 0
County                                              0
Latitude                                            0
Longitude                                           0
NERC Region                                         0
Balancing Authority Code                            0
Balancing Authority Name                            0
Name of Water Source                                0
Primary Purpose (NAICS Code)                        0
Regulatory Status                                   0
Sector                                              0
Sector Name                 

In [7]:
# drop fully empty columns
empty_cols = [c for c in df.columns if df[c].isna().all()]
df = df.drop(columns=empty_cols)

In [8]:
print(df['Primary Purpose (NAICS Code)'].value_counts())
print(df['FERC Cogeneration Status'].value_counts())
print(df['FERC Small Power Producer Status'].value_counts())
print(df['Energy Storage'].value_counts())
print(df['Natural Gas Pipeline Name 1'].value_counts())

Primary Purpose (NAICS Code)
22    56
Name: count, dtype: int64
FERC Cogeneration Status
N    56
Name: count, dtype: int64
FERC Small Power Producer Status
N    56
Name: count, dtype: int64
Energy Storage
N    56
Name: count, dtype: int64
Natural Gas Pipeline Name 1
FLORIDA GAS TRANSMISSION COMPANY    1
CAROLINA GAS TRANSMISSION CORP      1
Name: count, dtype: int64


In [9]:
# drop columns with all the same values for each row
df = df.drop(columns={'Primary Purpose (NAICS Code)', 'FERC Cogeneration Status', 'FERC Small Power Producer Status', 'Energy Storage', 'Natural Gas Pipeline Name 1'})

In [10]:
df.columns

Index(['Utility ID', 'Utility Name', 'Plant Code', 'Plant Name',
       'Street Address', 'City', 'State', 'Zip', 'County', 'Latitude',
       'Longitude', 'NERC Region', 'Balancing Authority Code',
       'Balancing Authority Name', 'Name of Water Source', 'Regulatory Status',
       'Sector', 'Sector Name', 'FERC Exempt Wholesale Generator Status',
       'FERC Exempt Wholesale Generator Docket Number', 'Ash Impoundment?',
       'Ash Impoundment Lined?', 'Transmission or Distribution System Owner',
       'Transmission or Distribution System Owner ID',
       'Transmission or Distribution System Owner State', 'Grid Voltage (kV)',
       'Grid Voltage 2 (kV)', 'Grid Voltage 3 (kV)', 'Natural Gas Storage',
       'Liquefied Natural Gas Storage'],
      dtype='object')

In [11]:
# keeping relevant columns
df = df[['Plant Name', 'County', 'Latitude', 'Longitude', 'Name of Water Source',
       'Regulatory Status', 'Grid Voltage (kV)']]

In [12]:
# spatial join with counties

county_path = Path.cwd().parent.joinpath('raw_data', 'county_boundaries_2025', 'tl_2025_us_county.shp')
# creating GeoDataFrame
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df['Longitude'], df['Latitude']), # creating Point geometry using 'lon' and 'lat' data values
    crs='EPSG:4326'
)

df_county = gpd.read_file(county_path)
if df_county.crs != gdf.crs: 
    df_county = df_county.to_crs(gdf.crs)

# creating new dataframe
gdf_plants = gpd.sjoin(gdf, df_county[['GEOID', 'NAMELSAD', 'geometry']],
                   how='left', predicate='within')
gdf_plants.head()

,Plant Name,County,Latitude,Longitude,Name of Water Source,Regulatory Status,Grid Voltage (kV),geometry,index_right,GEOID,NAMELSAD
0,Browns Ferry,Limestone,34.70420,-87.11890,Tennessee River,RE,500,POINT (-87.1189 34.7042),2798,01083,Limestone County
1,Clinton Power Station,DeWitt,40.17190,-88.83390,Salt Creek,NR,345,POINT (-88.8339 40.1719),1226,17039,De Witt County
2,Wolf Creek Generating Station,Coffey,38.23926,-95.68978,Wolf Creek Cooling Lake,RE,345,POINT (-95.68978 38.23926),464,20031,Coffey County
3,Columbia Generating Station,Benton,46.47110,-119.33390,Columbia River,RE,500,POINT (-119.3339 46.4711),3121,53005,Benton County
4,Millstone,New London,41.31070,-72.16770,Long Island Sound,NR,345,POINT (-72.1677 41.3107),1274,09180,Southeastern Connecticut Planning Region


In [13]:
# dropping and renaming columns
gdf_plants = gdf_plants.drop(columns=['index_right', 'geometry', 'Latitude', 'Longitude', 'County'])
gdf_plants = gdf_plants.rename(columns={'GEOID': 'geo_id', 'NAMELSAD': 'county_name'})

In [14]:
display(gdf_plants.head())
gdf_plants.shape

,Plant Name,Name of Water Source,Regulatory Status,Grid Voltage (kV),geo_id,county_name
0,Browns Ferry,Tennessee River,RE,500,01083,Limestone County
1,Clinton Power Station,Salt Creek,NR,345,17039,De Witt County
2,Wolf Creek Generating Station,Wolf Creek Cooling Lake,RE,345,20031,Coffey County
3,Columbia Generating Station,Columbia River,RE,500,53005,Benton County
4,Millstone,Long Island Sound,NR,345,09180,Southeastern Connecticut Planning Region


(56, 6)

In [15]:
# keeping relevant columns to be merged
# other columns could be used later and are stored in gdf_plants
power_plants = gdf_plants[['geo_id', 'county_name', 'Plant Name']]
power_plants.head()

,geo_id,county_name,Plant Name
0,01083,Limestone County,Browns Ferry
1,17039,De Witt County,Clinton Power Station
2,20031,Coffey County,Wolf Creek Generating Station
3,53005,Benton County,Columbia Generating Station
4,09180,Southeastern Connecticut Planning Region,Millstone


In [16]:
# aggregating by count
power_plants_agg = power_plants.groupby('geo_id').agg(county_name=('county_name', 'first'), plant_name=('Plant Name', lambda x: list(x.unique())), plant_count=('Plant Name', 'count')).reset_index()
power_plants_agg.head()

,geo_id,county_name,plant_name,plant_count
0,01069,Houston County,[Joseph M Farley],1
1,01083,Limestone County,[Browns Ferry],1
2,04013,Maricopa County,[Palo Verde],1
3,05115,Pope County,[Arkansas Nuclear One],1
4,06079,San Luis Obispo County,[Diablo Canyon],1


In [17]:
# saving as csv to 'processed' folder within 'data' folder
output_path = Path.cwd().parent.joinpath('processed_data', 'nuclear_plants_cleaned.csv')
output_path.parent.mkdir(parents=True, exist_ok=True)
power_plants_agg.to_csv(output_path, index=False)